# Week 65, Day 6: Forge: Week 65 challenge, quiz and deep read

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w065/d6.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

In [ ]:
print("1.", f"{5 ** 100:.3g} sequences; Viterbi needs about {100 * 5 * 5} operations")
print("2.", 2 / (2 + 0.5))
print("3. yes: 40 is far below the usual threshold of half the particles")

### Exercise: Part A: decoding head direction with particles

Complete the particle filter: propagate the angles by the random walk, weight by the Poisson log-likelihood of the observed counts, normalise in logs, and resample systematically when the effective sample size falls below half the particles.

In [ ]:
import numpy as np
from scipy.special import logsumexp
gA = np.random.default_rng(656)
T, N, P = 400, 20, 2000
pref = np.linspace(0, 2 * np.pi, N, endpoint=False)
kappa, peak, base = 3.0, 8.0, 0.2
sigma_walk = 0.12

ang = np.zeros(T)
for t in range(1, T):
    ang[t] = (ang[t - 1] + sigma_walk * gA.normal()) % (2 * np.pi)
rate = lambda a: base + peak * np.exp(kappa * (np.cos(a[..., None] - pref) - 1))
X = gA.poisson(rate(ang))

particles = gA.uniform(0, 2 * np.pi, P)
logw = np.full(P, -np.log(P))
est = np.zeros(T); resamples = 0
for t in range(T):
    particles = (particles + sigma_walk * gA.normal(size=P)) % (2 * np.pi)
    lam = rate(particles)                                            # P x N
    logw = logw + ...                                                # Poisson log-likelihood of X[t]
    logw -= logsumexp(logw)
    w = np.exp(logw)
    est[t] = np.angle(np.sum(w * np.exp(1j * particles)))
    ess = 1 / np.sum(w ** 2)
    if ess < P / 2:
        idx = gA.choice(P, P, p=w)
        particles = particles[idx]; logw = np.full(P, -np.log(P)); resamples += 1

circ_err = lambda a, b: np.mean(np.abs(np.angle(np.exp(1j * (a - b)))))
pv = np.angle(X @ np.exp(1j * pref))
err_pf, err_pv = circ_err(est, ang), circ_err(pv, ang)
print(f"circular error: particle filter {err_pf:.3f} rad, population vector {err_pv:.3f} rad; resampled on {resamples} of {T} bins")

In [ ]:
import numpy as np
def test_accurate():
    "the circular error of the particle filter is below 0.25 radians"
    assert err_pf < 0.25
def test_beats_pop():
    "it beats the population vector computed bin by bin"
    assert err_pf < err_pv - 0.02
def test_resample():
    "resampling happened on at least a tenth of the bins"
    assert resamples > T // 10

test_accurate()
test_beats_pop()
test_resample()
print("All checks passed")

### Exercise: Part B: how many states does the data want?

Generate one recording with two discrete rate levels and one with a smoothly drifting rate. Fit Poisson HMMs with 2 to 8 states on the first 700 bins and evaluate held-out log-likelihood per bin on the last 300, taking the smallest K within 0.02 of the best.

In [ ]:
import numpy as np
gB = np.random.default_rng(6561)
Tb, Nb = 1000, 12
load = gB.uniform(0.5, 1.5, Nb)

def make_discrete():
    A_ = np.array([[0.97, 0.03], [0.04, 0.96]]); s = np.zeros(Tb, int)
    for t in range(1, Tb): s[t] = gB.choice(2, p=A_[s[t - 1]])
    return gB.poisson(np.outer(np.where(s == 1, 4.0, 1.0), load))

def make_continuous():
    u = np.zeros(Tb)
    for t in range(1, Tb): u[t] = 0.98 * u[t - 1] + 0.2 * gB.normal()
    return gB.poisson(np.exp(np.outer(u, load) + np.log(2.0)))

def pois_emis(Xs, rates_):
    lg = Xs[:, None, :] * np.log(rates_)[None] - rates_[None]
    lg = lg.sum(2); lg -= lg.max(1, keepdims=True)
    return np.exp(lg) + 1e-300

def hmm_heldout(Xtr, Xte, K, iters=40):
    rates = np.outer(np.linspace(0.4, 2.5, K), Xtr.mean(0)) + 0.1
    A_ = np.full((K, K), 0.05 / max(K - 1, 1)); np.fill_diagonal(A_, 0.95); pi_ = np.full(K, 1 / K)
    for it in range(iters):
        B = pois_emis(Xtr, rates)
        al = np.zeros_like(B); c = np.zeros(len(B)); be = np.ones_like(B)
        a0 = pi_ * B[0]; c[0] = a0.sum(); al[0] = a0 / c[0]
        for t in range(1, len(B)):
            a0 = B[t] * (al[t - 1] @ A_); c[t] = a0.sum(); al[t] = a0 / c[t]
        for t in range(len(B) - 2, -1, -1):
            be[t] = (A_ @ (B[t + 1] * be[t + 1])) / c[t + 1]
        g = al * be; g /= g.sum(1, keepdims=True)
        xi = al[:-1, :, None] * A_[None] * (B[1:] * be[1:])[:, None, :]; xi /= xi.sum((1, 2), keepdims=True)
        A_ = xi.sum(0) / xi.sum(0).sum(1, keepdims=True); pi_ = g[0]
        rates = (g.T @ Xtr) / g.sum(0)[:, None] + 1e-6
    from scipy.stats import poisson as pois
    B = pois_emis(Xte, rates)
    scale = (pois.logpmf(Xte[:, None, :], rates[None]).sum(2) - np.log(B))[:, 0]   # restore the dropped constants
    al = np.zeros_like(B); c = np.zeros(len(B))
    a0 = pi_ * B[0]; c[0] = a0.sum(); al[0] = a0 / c[0]
    for t in range(1, len(B)):
        a0 = B[t] * (al[t - 1] @ A_); c[t] = a0.sum(); al[t] = a0 / c[t]
    return (np.sum(np.log(c)) + scale.sum()) / len(Xte)

Ks = [2, 3, 4, 5, 6, 8]
best_K = {}
for kind, Xall in [("discrete", make_discrete()), ("continuous", make_continuous())]:
    Xtr, Xte = Xall[:700], Xall[700:]
    ho = {K: hmm_heldout(Xtr, Xte, K) for K in Ks}
    top = max(ho.values())
    best_K[kind] = ...
    print(f"{kind:11s}", {K: round(float(v), 3) for K, v in ho.items()}, "-> K =", best_K[kind])

In [ ]:
import numpy as np
def test_discrete():
    "for two-level data, held-out likelihood stops improving after 2 states"
    assert best_K["discrete"] <= 3
def test_continuous():
    "a smoothly drifting latent needs more states than a two-level one"
    assert best_K["continuous"] >= best_K["discrete"] + 2

test_discrete()
test_continuous()
print("All checks passed")